# Armazenamento de Dados
### Guia prático com Python 3 e Django 6

**Do primeiro cadastro em memória à persistência em banco de dados.**

**Público:** iniciantes e estudantes de nível intermediário  
**Projeto contínuo:** controle de estoque de uma pequena loja  
**Formato:** explicação → exemplo comentado → resultado → prática

> **Como estudar:** execute as células na ordem, usando **Shift + Enter**. Os exemplos das seções de Python, arquivos e SQLite funcionam apenas com a biblioteca padrão. A seção de Django 6 é opcional e exige instalação específica.

---

**Roteiro do notebook**

1. [O que é armazenamento de dados?](#1-o-que-é-armazenamento-de-dados)
2. [Características](#2-características)
3. [Quando utilizar cada opção](#3-quando-utilizar-cada-opção)
4. [Principais métodos — na prática](#4-principais-métodos--na-prática)
   - [4.1 Memória: listas e dicionários](#41-armazenamento-em-memória)
   - [4.2 TXT: texto puro](#42-arquivos-txt)
   - [4.3 JSON: dados estruturados](#43-arquivos-json)
   - [4.4 CSV: dados em tabela](#44-arquivos-csv)
   - [4.5 SQLite: banco de dados SQL](#45-sqlite--banco-de-dados-relacional)
   - [4.6 Django 6: ORM](#46-django-6--orm)
5. [Desafio prático e verificação](#5-desafio-prático-e-verificação)
6. [Boas práticas, erros comuns e referências](#6-boas-práticas-e-referências)

## Objetivos de aprendizagem

Ao concluir esta aula, você conseguirá:

- Diferenciar **dados em memória** de **dados persistentes**.
- Salvar, consultar e atualizar produtos usando **TXT, JSON e CSV**.
- Executar o ciclo **CRUD** em um banco SQLite.
- Reconhecer os principais métodos do **ORM do Django 6**.
- Escolher a forma de armazenamento apropriada para cada situação.

**Observação sobre valores monetários:** usaremos `preco_centavos` como número inteiro. Por exemplo, `2590` representa **R$ 25,90**. Isso evita erros de arredondamento associados a cálculos financeiros com `float`. Em outros projetos, `decimal.Decimal` também é uma opção adequada.

## 1. O que é armazenamento de dados?

**Armazenar dados** significa guardar informações para que um programa possa utilizá-las agora ou posteriormente.

Imagine uma loja com produtos, preços e quantidades em estoque:

| Campo | Exemplo | Significado |
|---|---|---|
| `id` | `1` | Identificador do produto |
| `nome` | `"Caderno Universitário"` | Descrição |
| `preco_centavos` | `2590` | Preço: R$ 25,90 |
| `estoque` | `10` | Unidades disponíveis |

### Duas ideias essenciais

- **Memória (RAM):** objetos Python existem enquanto o programa está em execução. Ao encerrar o processo, esses objetos normalmente desaparecem.
- **Persistência:** um arquivo ou banco de dados permite recuperar os dados em outra execução do programa.

**Exemplo cotidiano:** uma lista Python é como anotar itens num quadro temporário; um arquivo ou banco de dados é como registrar os itens num caderno que pode ser consultado depois.

### O que é CRUD?

| Sigla | Ação | Exemplo na loja |
|---|---|---|
| **C — Create** | Criar | Cadastrar um produto |
| **R — Read** | Ler | Consultar os produtos |
| **U — Update** | Atualizar | Alterar o estoque |
| **D — Delete** | Excluir | Remover um produto |

Você verá essas quatro operações reaparecendo em diferentes tecnologias.

### Preparação do projeto de exemplo

Todos os exercícios usam o **mesmo catálogo inicial**. Os arquivos gerados ficam na pasta local `dados_aula/`, ao lado do notebook. Não utilize nela dados reais ou importantes: alguns exemplos sobrescrevem seus arquivos de treinamento.

In [1]:
# Importamos ferramentas que já acompanham o Python.
from pathlib import Path
from copy import deepcopy

# Pasta onde salvaremos os arquivos e o banco SQLite desta aula.
PASTA_DADOS = Path('dados_aula')
PASTA_DADOS.mkdir(exist_ok=True)

# Catálogo único que servirá de referência em todas as demonstrações.
produtos_base = [
    {'id': 1, 'nome': 'Caderno Universitário', 'preco_centavos': 2590, 'estoque': 10},
    {'id': 2, 'nome': 'Caneta Azul', 'preco_centavos': 350, 'estoque': 40},
    {'id': 3, 'nome': 'Mochila Escolar', 'preco_centavos': 8990, 'estoque': 5},
]

# Converte um preço expresso em centavos para uma apresentação em reais.
def formatar_reais(centavos):
    valor = centavos / 100
    return f'R$ {valor:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# Mostra o catálogo de maneira legível.
for produto in produtos_base:
    print(f"#{produto['id']} | {produto['nome']} | "
          f"{formatar_reais(produto['preco_centavos'])} | "
          f"Estoque: {produto['estoque']}")

#1 | Caderno Universitário | R$ 25,90 | Estoque: 10
#2 | Caneta Azul | R$ 3,50 | Estoque: 40
#3 | Mochila Escolar | R$ 89,90 | Estoque: 5


## 2. Características

Nem toda forma de armazenamento oferece as mesmas vantagens. Antes de escolher, considere:

| Característica | Pergunta importante | Exemplo |
|---|---|---|
| **Persistência** | O dado continuará disponível após fechar o programa? | `JSON`: sim; lista: não, por si só |
| **Estrutura** | Existem campos e tipos organizados? | Banco SQL define colunas; TXT é mais livre |
| **Facilidade de leitura** | Uma pessoa consegue abrir e entender o conteúdo? | TXT, CSV e JSON são legíveis |
| **Consultas** | Preciso buscar, ordenar e filtrar rapidamente? | Banco de dados facilita consultas estruturadas |
| **Concorrência** | Mais de um usuário alterará os dados? | Bancos costumam ser mais apropriados |
| **Integridade** | Como evitar dados inválidos ou inconsistentes? | `CHECK`, tipos, chaves e transações |
| **Segurança** | Quem pode ler e alterar o conteúdo? | Permissões, autenticação e configurações |
| **Escalabilidade** | A quantidade de dados poderá crescer? | PostgreSQL é comum em aplicações maiores |

> **Importante:** arquivos são úteis e simples, mas normalmente exigem que o próprio programa controle regravações, validações e conflitos de edição.

## 3. Quando utilizar cada opção

| Solução | Quando utilizar | Limitação principal |
|---|---|---|
| **Lista / dicionário** | Dados temporários, cálculos e protótipos | Não persistem automaticamente |
| **TXT** | Anotações, logs didáticos e conteúdo textual simples | Não tem estrutura de consulta própria |
| **JSON** | Configurações, integração com APIs, pequenas coleções | Atualização geralmente exige regravar o documento |
| **CSV** | Planilhas, exportações e importações tabulares | Tipos são lidos como texto; relações são limitadas |
| **SQLite** | Projetos locais, estudos, aplicações de pequeno porte | Escrita concorrente é mais limitada |
| **Django ORM + banco** | Aplicações web com modelos, regras e consultas | Requer configurar projeto e banco |

**Escolha prática:** para aprender CRUD sem servidor de banco, comece por **SQLite**. Em projetos Django, use **models e ORM**, com um banco compatível (por exemplo, SQLite no aprendizado e PostgreSQL quando fizer sentido na implantação).

**Atenção:** Django ORM não é um banco de dados: é uma interface Python para trabalhar com um banco configurado no projeto.

## 4. Principais métodos — na prática

A partir de agora, os exemplos mostrarão **como criar, consultar, atualizar e excluir** informações do mesmo estoque de produtos.

### 4.1 Armazenamento em memória

**O que é:** objetos Python, como `list` e `dict`, armazenados em memória durante a execução.

**Características:** muito rápido e flexível, mas sem persistência automática.

**Quando utilizar:** listas de resultados, cálculos, testes, pré-processamento e pequenas rotinas temporárias.

**Principais métodos:** `append()`, `get()`, `update()` e `pop()`.

#### Método `list.append()` — adicionar um produto

In [2]:
# deepcopy cria uma cópia independente, preservando nosso catálogo original.
produtos_memoria = deepcopy(produtos_base)

# append() insere um novo dicionário ao final da lista.
produtos_memoria.append({
    'id': 4, 'nome': 'Régua 30 cm', 'preco_centavos': 450, 'estoque': 20
})

print('Produtos em memória:', len(produtos_memoria))
print('Último cadastrado:', produtos_memoria[-1]['nome'])

Produtos em memória: 4
Último cadastrado: Régua 30 cm


#### Método `dict.get()` — ler um campo com valor padrão

In [3]:
# Encontramos o produto de id 2 dentro da lista.
caneta = next(p for p in produtos_memoria if p['id'] == 2)

# get() devolve o campo informado ou um valor padrão se ele não existir.
print('Produto:', caneta.get('nome', 'Sem nome'))
print('Categoria:', caneta.get('categoria', 'Não informada'))

Produto: Caneta Azul
Categoria: Não informada


#### Método `dict.update()` — atualizar campos

In [4]:
# Atualizamos apenas o estoque do primeiro produto da lista.
caderno = next(p for p in produtos_memoria if p['id'] == 1)
caderno.update({'estoque': 12})

print('Novo estoque do caderno:', caderno['estoque'])
print('Catálogo original preservado:', produtos_base[0]['estoque'])

Novo estoque do caderno: 12
Catálogo original preservado: 10


#### Método `list.pop()` — remover um produto pela posição

In [5]:
# pop() remove e devolve o item da posição informada.
# Como a régua foi adicionada ao fim, usamos -1 (último índice).
removido = produtos_memoria.pop(-1)
print('Removido:', removido['nome'])
print('Produtos restantes:', len(produtos_memoria))

Removido: Régua 30 cm
Produtos restantes: 3


### 4.2 Arquivos TXT

**O que é:** um arquivo de texto simples, como `inventario.txt`.

**Características:** fácil de abrir e entender; não define campos ou relacionamentos automaticamente.

**Quando utilizar:** observações, pequenos relatórios, mensagens e registros textuais.

**Principais métodos:** `Path.write_text()`, `Path.read_text()` e `Path.open()`.

> Em arquivos: modo `w` **substitui** o conteúdo anterior; `a` **acrescenta** ao final; `r` **lê**. Especificar `encoding='utf-8'` ajuda a preservar acentos.

#### Método `write_text()` — criar ou substituir um TXT

In [6]:
CAMINHO_TXT = PASTA_DADOS / 'inventario.txt'

# Montamos linhas de texto a partir do catálogo.
linhas = [f"{p['id']} | {p['nome']} | {p['estoque']} unidades"
          for p in produtos_base]
conteudo = 'INVENTÁRIO DA LOJA\n' + '\n'.join(linhas) + '\n'

# write_text() cria o arquivo ou sobrescreve seu conteúdo existente.
CAMINHO_TXT.write_text(conteudo, encoding='utf-8')
print('Arquivo TXT salvo em:', CAMINHO_TXT)

Arquivo TXT salvo em: dados_aula/inventario.txt


#### Método `read_text()` — ler o conteúdo completo

In [7]:
# read_text() devolve todo o conteúdo do arquivo como uma string.
texto_lido = CAMINHO_TXT.read_text(encoding='utf-8')
print(texto_lido)

INVENTÁRIO DA LOJA
1 | Caderno Universitário | 10 unidades
2 | Caneta Azul | 40 unidades
3 | Mochila Escolar | 5 unidades



#### Método `open()` — acrescentar uma observação

In [8]:
# O modo 'a' (append) preserva o conteúdo e adiciona uma nova linha.
with CAMINHO_TXT.open('a', encoding='utf-8') as arquivo:
    arquivo.write('Observação: conferir o estoque ao final do dia.\n')

# Exibimos apenas a última linha para verificar a inclusão.
print(CAMINHO_TXT.read_text(encoding='utf-8').splitlines()[-1])

Observação: conferir o estoque ao final do dia.


### 4.3 Arquivos JSON

**O que é:** formato de texto para dados estruturados. É muito usado em APIs e configurações.

**Características:** representa listas, objetos, textos, números, booleanos e valores nulos.

**Quando utilizar:** trocar dados entre programas, armazenar configurações e pequenos conjuntos de registros.

**Principais funções:** `json.dump()`, `json.load()`, `json.dumps()` e `json.loads()`.

**Diferença importante:** `dump/load` trabalham com **arquivo**; `dumps/loads` trabalham com **string**. Em JSON, as chaves de objetos são strings e tipos Python não suportados diretamente precisam de conversão.

#### Método `json.dump()` — gravar os produtos em arquivo

In [9]:
import json

CAMINHO_JSON = PASTA_DADOS / 'produtos.json'

# dump() converte a lista de dicionários em JSON e grava no arquivo.
with CAMINHO_JSON.open('w', encoding='utf-8') as arquivo:
    json.dump(produtos_base, arquivo, ensure_ascii=False, indent=4)

print('JSON salvo em:', CAMINHO_JSON)

JSON salvo em: dados_aula/produtos.json


#### Método `json.load()` — ler os produtos de volta

In [10]:
# load() recupera a estrutura: aqui, uma lista de dicionários Python.
with CAMINHO_JSON.open('r', encoding='utf-8') as arquivo:
    produtos_json = json.load(arquivo)

print('Quantidade de produtos:', len(produtos_json))
print('Primeiro produto:', produtos_json[0]['nome'])

Quantidade de produtos: 3
Primeiro produto: Caderno Universitário


#### Atualização de um JSON — `load()` + alteração + `dump()`

In [11]:
# Mudamos o estoque da caneta de 40 para 37 unidades.
produto_json = next(p for p in produtos_json if p['id'] == 2)
produto_json['estoque'] = 37

# JSON não possui uma operação nativa para atualizar um registro do arquivo:
# gravamos novamente a lista completa com os dados modificados.
with CAMINHO_JSON.open('w', encoding='utf-8') as arquivo:
    json.dump(produtos_json, arquivo, ensure_ascii=False, indent=4)

print('Novo estoque salvo no JSON:', produto_json['estoque'])

Novo estoque salvo no JSON: 37


#### Métodos `json.dumps()` e `json.loads()` — converter sem salvar arquivo

In [12]:
# dumps() serializa um dicionário em texto JSON (string).
texto_json = json.dumps(produtos_json[0], ensure_ascii=False)

# loads() converte a string JSON de volta para um objeto Python.
produto_recuperado = json.loads(texto_json)

print('Tipo serializado:', type(texto_json).__name__)
print('Nome recuperado:', produto_recuperado['nome'])

Tipo serializado: str
Nome recuperado: Caderno Universitário


### 4.4 Arquivos CSV

**O que é:** arquivo de tabela em que cada linha corresponde a um registro e as colunas guardam valores.

**Características:** pode ser aberto por editores de planilhas; é adequado para importações/exportações.

**Quando utilizar:** relatórios tabulares, integração com planilhas e transferências simples entre sistemas.

**Principais recursos:** `csv.DictWriter`, `writeheader()`, `writerows()` e `csv.DictReader`.

> **Atenção:** `DictReader` lê os valores como **strings**. Converta números com `int()` ou outra função quando necessário. Aqui usamos `delimiter=';'`, comum em CSVs usados com planilhas em português.

#### `DictWriter`, `writeheader()` e `writerows()` — criar a tabela

In [13]:
import csv

CAMINHO_CSV = PASTA_DADOS / 'produtos.csv'
COLUNAS_CSV = ['id', 'nome', 'preco_centavos', 'estoque']

# newline='' evita linhas extras em diferentes sistemas operacionais.
with CAMINHO_CSV.open('w', encoding='utf-8', newline='') as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=COLUNAS_CSV, delimiter=';')
    escritor.writeheader()              # Escreve o título das colunas.
    escritor.writerows(produtos_base)   # Escreve um produto por linha.

print('CSV salvo em:', CAMINHO_CSV)

CSV salvo em: dados_aula/produtos.csv


#### `DictReader` — ler linhas e recuperar tipos numéricos

In [14]:
# DictReader associa automaticamente cada coluna ao seu cabeçalho.
with CAMINHO_CSV.open('r', encoding='utf-8', newline='') as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=';')
    produtos_csv = [
        {
            'id': int(linha['id']),
            'nome': linha['nome'],
            'preco_centavos': int(linha['preco_centavos']),
            'estoque': int(linha['estoque']),
        }
        for linha in leitor
    ]

print('Tipo de id após conversão:', type(produtos_csv[0]['id']).__name__)
print('Produtos lidos do CSV:', len(produtos_csv))

Tipo de id após conversão: int
Produtos lidos do CSV: 3


#### Atualização de CSV — ler, alterar e regravar

In [15]:
# O estoque da caneta também será atualizado para 37 unidades.
next(p for p in produtos_csv if p['id'] == 2)['estoque'] = 37

# Para editar registros de modo simples, reescrevemos o CSV inteiro.
with CAMINHO_CSV.open('w', encoding='utf-8', newline='') as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=COLUNAS_CSV, delimiter=';')
    escritor.writeheader()
    escritor.writerows(produtos_csv)

print('Novo estoque da caneta no CSV:', produtos_csv[1]['estoque'])

Novo estoque da caneta no CSV: 37


### 4.5 SQLite — banco de dados relacional

**O que é:** um banco de dados SQL que fica em um arquivo no computador, sem precisar instalar um servidor separado.

**Características:** permite consultar registros por condições, armazená-los em tabelas e usar transações.

**Quando utilizar:** aplicações locais, ferramentas pequenas, protótipos, estudos e projetos de desenvolvimento.

**Principais métodos e comandos:** `sqlite3.connect()`, `execute()`, `executemany()`, `fetchone()`, `fetchall()`, `commit()` e `rollback()`; comandos SQL `INSERT`, `SELECT`, `UPDATE` e `DELETE`.

**Boa prática de segurança:** dados externos devem ser enviados como **parâmetros** com `?`, em vez de concatenados diretamente dentro de comandos SQL.

#### `connect()` e `execute()` — conectar e criar a tabela

In [16]:
import sqlite3
from contextlib import closing

CAMINHO_BANCO = PASTA_DADOS / 'loja_exemplo.sqlite3'

# connect() abre ou cria um banco SQLite no caminho informado.
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # execute() envia um comando SQL ao banco.
    conexao.execute('''
        CREATE TABLE IF NOT EXISTS produtos (
            id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            preco_centavos INTEGER NOT NULL CHECK (preco_centavos >= 0),
            estoque INTEGER NOT NULL CHECK (estoque >= 0)
        )
    ''')

    # Reiniciamos SOMENTE a tabela de treinamento para repetir a aula.
    conexao.execute('DELETE FROM produtos')
    conexao.commit()  # Confirma as alterações realizadas nesta conexão.

print('Banco preparado:', CAMINHO_BANCO)

Banco preparado: dados_aula/loja_exemplo.sqlite3


#### `executemany()` + `INSERT` + `commit()` — cadastrar os produtos

In [17]:
# Convertemos os dicionários em tuplas na mesma ordem das colunas SQL.
registros = [
    (p['id'], p['nome'], p['preco_centavos'], p['estoque'])
    for p in produtos_base
]

with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # Os quatro sinais '?' recebem valores de cada tupla, em segurança.
    conexao.executemany('''
        INSERT INTO produtos (id, nome, preco_centavos, estoque)
        VALUES (?, ?, ?, ?)
    ''', registros)
    conexao.commit()  # Sem confirmar a transação, os dados podem não persistir.

print('Produtos cadastrados no SQLite:', len(registros))

Produtos cadastrados no SQLite: 3


#### `execute()` + `fetchall()` — listar produtos

In [18]:
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # SELECT seleciona os campos; ORDER BY organiza pelo identificador.
    cursor = conexao.execute('''
        SELECT id, nome, preco_centavos, estoque
        FROM produtos
        ORDER BY id
    ''')
    linhas_banco = cursor.fetchall()  # Lê todas as linhas resultantes.

for id_produto, nome, preco, estoque in linhas_banco:
    print(f'{id_produto}: {nome} — {formatar_reais(preco)} — {estoque} un.')

1: Caderno Universitário — R$ 25,90 — 10 un.
2: Caneta Azul — R$ 3,50 — 40 un.
3: Mochila Escolar — R$ 89,90 — 5 un.


#### `fetchone()` + `WHERE` — consultar um produto

In [19]:
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # Passamos o id como parâmetro: (2,) é uma tupla de um elemento.
    cursor = conexao.execute(
        'SELECT nome, estoque FROM produtos WHERE id = ?', (2,)
    )
    linha = cursor.fetchone()  # Lê só o primeiro resultado ou None.

print('Produto encontrado:', linha)

Produto encontrado: ('Caneta Azul', 40)


#### `UPDATE` — alterar o estoque no banco

In [20]:
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # Alteramos somente o registro cujo id é 2.
    cursor = conexao.execute(
        'UPDATE produtos SET estoque = ? WHERE id = ?', (37, 2)
    )
    conexao.commit()
    print('Registros atualizados:', cursor.rowcount)

Registros atualizados: 1


#### `DELETE` — excluir um produto

In [21]:
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    # Apagamos o produto de id 3 (Mochila Escolar).
    cursor = conexao.execute('DELETE FROM produtos WHERE id = ?', (3,))
    conexao.commit()
    print('Registros excluídos:', cursor.rowcount)

Registros excluídos: 1


#### `rollback()` — desfazer uma alteração antes de confirmar

In [22]:
# Simulamos uma alteração que não deve ser concluída.
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    try:
        conexao.execute('UPDATE produtos SET estoque = ? WHERE id = ?', (0, 1))
        # Imagine que uma validação de negócio falhou.
        raise ValueError('Estoque não deve ser zerado neste exemplo.')
    except ValueError as erro:
        conexao.rollback()  # Descarta a alteração ainda não confirmada.
        print('Operação cancelada:', erro)

    cursor = conexao.execute('SELECT estoque FROM produtos WHERE id = ?', (1,))
    print('Estoque após rollback:', cursor.fetchone()[0])

Operação cancelada: Estoque não deve ser zerado neste exemplo.
Estoque após rollback: 10


### 4.6 Django 6 — ORM

**O que é:** o *Object-Relational Mapper* (ORM) do Django permite trabalhar com tabelas usando classes e métodos Python, sem escrever SQL para cada operação.

**Características:** models, consultas por campos, validações possíveis, migrações e integração com a aplicação web.

**Quando utilizar:** quando o projeto Django precisa persistir informações da aplicação (produtos, clientes, pedidos etc.).

**Principais métodos:** `create()`, `all()`, `filter()`, `get()`, `get_or_create()`, `order_by()`, `count()`, `exists()`, `update()`, `save()` e `delete()`.

**Requisitos:** Django **6.0.x** funciona com Python **3.12, 3.13 ou 3.14**. [Documentação oficial do Django 6.0](https://docs.djangoproject.com/en/6.0/releases/6.0/).

> **Importante:** a primeira demonstração abaixo é um **laboratório isolado para notebook** e cria uma tabela exclusivamente didática. Em um projeto Django de verdade, defina modelos em `models.py` e crie as tabelas usando `makemigrations` e `migrate`, como explicado adiante.

#### Preparação opcional do Django no Jupyter

Se o pacote não estiver instalado no ambiente do seu notebook, execute a seguinte linha em **uma célula nova**, uma única vez, e depois reinicie o kernel:

```python
%pip install "Django>=6.0,<6.1"
```

O código a seguir **não instala pacotes automaticamente**. Se o Django estiver ausente, ele informa a situação, e os exercícios do ORM serão pulados sem impedir a execução das outras seções.

**Não execute o laboratório isolado dentro de um kernel que já configurou um projeto Django existente:** utilize o shell do seu projeto nesse caso.

#### Laboratório isolado: preparar o banco e o model para os exemplos do ORM

In [23]:
# Este bloco permite executar o notebook mesmo quando Django não está instalado.
django_ativo = False

try:
    import django
except ImportError:
    print('Django não está instalado. Para ativar o laboratório, use:')
    print('  %pip install "Django>=6.0,<6.1"')
    print('Depois reinicie o kernel e execute o notebook novamente.')
else:
    from django.conf import settings

    if django.VERSION[:2] != (6, 0):
        print('Django detectado:', django.get_version())
        print('Este laboratório foi preparado para Django 6.0.x.')
    elif settings.configured:
        print('O Django já está configurado neste kernel.')
        print('Para evitar alterar outro projeto, abra este notebook em um kernel limpo.')
    else:
        # Banco independente do SQLite usado na seção anterior.
        CAMINHO_DJANGO = str(PASTA_DADOS / 'loja_django_notebook.sqlite3')
        settings.configure(
            SECRET_KEY='apenas-aula-nao-utilizar-em-producao',
            INSTALLED_APPS=[],
            DATABASES={
                'default': {'ENGINE': 'django.db.backends.sqlite3', 'NAME': CAMINHO_DJANGO}
            },
            DEFAULT_AUTO_FIELD='django.db.models.BigAutoField',
        )
        django.setup()

        from django.db import models, connection

        # Um model representa os campos da tabela de produtos.
        class ProdutoORM(models.Model):
            nome = models.CharField(max_length=100)
            preco_centavos = models.PositiveIntegerField()
            estoque = models.PositiveIntegerField()

            class Meta:
                app_label = 'aula_dados'  # Rótulo do exemplo sem app instalada.
                db_table = 'produtos_notebook'

        # Apenas nesta aula, criamos a tabela diretamente (sem migrations).
        # Em um projeto real, use makemigrations e migrate.
        if ProdutoORM._meta.db_table not in connection.introspection.table_names():
            with connection.schema_editor() as editor:
                editor.create_model(ProdutoORM)

        # Reset somente da tabela do laboratório para resultados previsíveis.
        ProdutoORM.objects.all().delete()
        django_ativo = True
        print('Django', django.get_version(), '| laboratório preparado com sucesso.')

Django não está instalado. Para ativar o laboratório, use:
  %pip install "Django>=6.0,<6.1"
Depois reinicie o kernel e execute o notebook novamente.


#### Método `objects.create()` — criar registros

In [24]:
if django_ativo:
    # create() cria e salva cada objeto no banco de dados.
    for p in produtos_base:
        ProdutoORM.objects.create(
            id=p['id'],
            nome=p['nome'],
            preco_centavos=p['preco_centavos'],
            estoque=p['estoque'],
        )
    print('Produtos cadastrados:', ProdutoORM.objects.count())
else:
    print('Exemplo não executado: configure o Django 6.0 para praticar.')

Exemplo não executado: configure o Django 6.0 para praticar.


#### Método `objects.all()` — consultar todos os registros

In [25]:
if django_ativo:
    # all() produz um QuerySet contendo todos os produtos da tabela.
    for produto in ProdutoORM.objects.all():
        print(produto.id, '|', produto.nome)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Métodos `filter()` e `order_by()` — filtrar e ordenar

In [26]:
if django_ativo:
    # filter() pode devolver zero, um ou vários produtos.
    # icontains busca parte do texto sem diferenciar maiúsculas/minúsculas.
    resultado = ProdutoORM.objects.filter(nome__icontains='a').order_by('nome')
    print('Produtos com a letra A no nome:')
    for produto in resultado:
        print('-', produto.nome)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Método `get()` — buscar exatamente um produto

In [27]:
if django_ativo:
    # get() é apropriado quando esperamos exatamente um resultado.
    # Ele gera exceção se não existir ou se houver mais de um correspondente.
    produto = ProdutoORM.objects.get(pk=1)
    print('Localizado:', produto.nome)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Método `get_or_create()` — recuperar ou cadastrar

In [28]:
if django_ativo:
    # Procura a régua pelo id; se não existir, cria com os defaults.
    regua, foi_criado = ProdutoORM.objects.get_or_create(
        id=4,
        defaults={'nome': 'Régua 30 cm', 'preco_centavos': 450, 'estoque': 20},
    )
    print('Produto:', regua.nome, '| Foi criado?', foi_criado)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Método `update()` — alterar registros diretamente na consulta

In [29]:
if django_ativo:
    # update() altera as linhas selecionadas e informa quantas foram afetadas.
    alterados = ProdutoORM.objects.filter(pk=2).update(estoque=37)
    print('Linhas alteradas:', alterados)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Método `save()` — salvar alterações em uma instância

In [30]:
if django_ativo:
    # Modificamos o objeto Python e persistimos a mudança com save().
    caderno_orm = ProdutoORM.objects.get(pk=1)
    caderno_orm.estoque = 12
    caderno_orm.save(update_fields=['estoque'])
    print('Estoque persistido:', ProdutoORM.objects.get(pk=1).estoque)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Métodos `count()` e `exists()` — contar e verificar existência

In [31]:
if django_ativo:
    # count() retorna a quantidade de registros da consulta.
    print('Total de produtos:', ProdutoORM.objects.count())

    # exists() confirma se há pelo menos um registro correspondente.
    ha_caneta = ProdutoORM.objects.filter(nome='Caneta Azul').exists()
    print('Existe caneta azul?', ha_caneta)
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Método `delete()` — excluir o registro selecionado

In [32]:
if django_ativo:
    # Excluímos a mochila, identificada pelo id 3.
    quantidade, detalhes = ProdutoORM.objects.filter(pk=3).delete()
    print('Objetos excluídos:', quantidade)
    print('Produtos restantes:', ProdutoORM.objects.count())
else:
    print('Exemplo do ORM pulado.')

Exemplo do ORM pulado.


#### Como fazer no seu projeto Django real

O laboratório acima demonstra métodos do ORM no notebook; no projeto real, você deve usar uma **app Django** e **migrações**.

**Exemplo de arquivo `loja/models.py`:**

```python
from django.db import models

class Produto(models.Model):
    # Campos da tabela produtos.
    nome = models.CharField(max_length=100)
    preco_centavos = models.PositiveIntegerField()
    estoque = models.PositiveIntegerField(default=0)

    def __str__(self):
        # Texto exibido no admin e em outras representações.
        return self.nome
```

**Fluxo do projeto (no terminal, dentro da pasta com `manage.py`):**

```bash
# Somente caso a app 'loja' ainda não exista:
python manage.py startapp loja

# Adicione 'loja' a INSTALLED_APPS em settings.py.
# Então gere as migrações a partir dos models:
python manage.py makemigrations

# Crie/atualize as tabelas no banco configurado:
python manage.py migrate

# Abra o shell para usar os models:
python manage.py shell
```

**No shell Python do Django:**

```python
from loja.models import Produto

# Cadastrar um produto na base de dados.
Produto.objects.create(nome='Caderno Universitário', preco_centavos=2590, estoque=10)

# Consultar produtos com estoque disponível.
print(list(Produto.objects.filter(estoque__gt=0).values('nome', 'estoque')))
```

**Nota de nível intermediário:** `QuerySet.update()` faz uma atualização direta no banco, sem chamar o método `save()` de cada instância. Para operações relacionadas que precisam ser confirmadas ou desfeitas em conjunto, estude `transaction.atomic()`.

## 5. Desafio prático e verificação

### Desafio: qual é o valor do estoque atual?

No nosso banco SQLite, há dois produtos restantes depois da exclusão da mochila. Descubra o **valor total em estoque**, somando `preco_centavos × estoque`.

**Tente antes de olhar a solução:** escreva uma consulta com `SELECT SUM(...) FROM produtos`. Depois converta os centavos para reais com `formatar_reais()`.

In [33]:
# Solução: o próprio SQL calcula o valor total de todas as unidades.
with closing(sqlite3.connect(CAMINHO_BANCO)) as conexao:
    cursor = conexao.execute('''
        SELECT COALESCE(SUM(preco_centavos * estoque), 0)
        FROM produtos
    ''')
    valor_total_centavos = cursor.fetchone()[0]

print('Valor total do estoque SQLite:', formatar_reais(valor_total_centavos))

Valor total do estoque SQLite: R$ 388,50


### Autoverificação

As instruções `assert` abaixo verificam os resultados principais. Se alguma falhar, execute novamente o notebook do início, na ordem, e verifique qual célula foi modificada.

In [34]:
# Nossa lista original não foi alterada pela cópia usada nas experiências.
assert len(produtos_base) == 3
assert produtos_base[0]['estoque'] == 10

# JSON e CSV foram atualizados para estoque de 37 unidades de caneta.
with CAMINHO_JSON.open('r', encoding='utf-8') as arquivo:
    dados_json_finais = json.load(arquivo)
assert next(p for p in dados_json_finais if p['id'] == 2)['estoque'] == 37

with CAMINHO_CSV.open('r', encoding='utf-8', newline='') as arquivo:
    dados_csv_finais = list(csv.DictReader(arquivo, delimiter=';'))
assert next(p for p in dados_csv_finais if int(p['id']) == 2)['estoque'] == '37'

# SQLite mantém caderno (10 unidades) e caneta (37 unidades).
# 10 × R$ 25,90 + 37 × R$ 3,50 = R$ 388,50.
assert valor_total_centavos == 38850

if django_ativo:
    # Os exemplos do ORM terminam com 3 produtos (incluindo a régua).
    assert ProdutoORM.objects.count() == 3
    assert ProdutoORM.objects.get(pk=1).estoque == 12

print('✓ Verificações concluídas com sucesso!')

✓ Verificações concluídas com sucesso!


## 6. Boas práticas e referências

### Boas práticas para projetos reais

1. **Separe o formato da finalidade:** memória é transitória; TXT/JSON/CSV são arquivos; SQLite/PostgreSQL são bancos; Django ORM é uma camada de acesso ao banco.
2. **Nunca monte SQL concatenando entradas do usuário:** prefira parâmetros (`?` no `sqlite3`) ou consultas pelo ORM.
3. **Valide o que receber:** campos obrigatórios, tipos, estoque não negativo, identificadores únicos etc.
4. **Proteja informações sensíveis:** não coloque senhas ou tokens em arquivos de exemplo nem publique dados privados em repositórios.
5. **Trate falhas e faça backup:** arquivo salvo não equivale a cópia de segurança; planeje recuperação de dados.
6. **Evite perda de dados:** `w`, `write_text()`, regravação de JSON/CSV e `DELETE` podem substituir/remover registros.
7. **Organize os dados em models Django:** em aplicações reais, controle alterações da estrutura usando migrações e arquivos de configuração adequados.

### Erros comuns e como pensar neles

| Erro | Possível causa | Como resolver |
|---|---|---|
| `FileNotFoundError` | Caminho incorreto ou arquivo ainda não criado | Execute a célula que grava primeiro |
| `json.JSONDecodeError` | JSON inválido ou parcialmente escrito | Confira o conteúdo e a serialização |
| `ValueError` ao usar `int()` | CSV contém texto incompatível com número | Valide os dados antes da conversão |
| `sqlite3.IntegrityError` | Chave repetida ou restrição violada | Verifique IDs, colunas e valores |
| `django.core.exceptions.AppRegistryNotReady` | Django ainda não foi preparado | Configure a aplicação e execute `django.setup()` quando apropriado |
| `OperationalError: no such table` | Tabela ou migrações ausentes | Execute a preparação do banco ou as migrações |

### Referências oficiais (leitura complementar)

- [Python: operações de entrada/saída e arquivos](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)
- [Python: módulo `json`](https://docs.python.org/3/library/json.html)
- [Python: módulo `csv`](https://docs.python.org/3/library/csv.html)
- [Python: módulo `sqlite3`](https://docs.python.org/3/library/sqlite3.html)
- [Django 6.0: models](https://docs.djangoproject.com/en/6.0/topics/db/models/)
- [Django 6.0: consultas de banco](https://docs.djangoproject.com/en/6.0/topics/db/queries/)
- [Django 6.0: API de QuerySet](https://docs.djangoproject.com/en/6.0/ref/models/querysets/)
- [Django 6.0: notas de versão e compatibilidade](https://docs.djangoproject.com/en/6.0/releases/6.0/)

---

### Em resumo

**Memória** para processamento temporário; **TXT** para texto simples; **JSON** para objetos estruturados; **CSV** para tabelas intercambiáveis; **SQLite** para um banco local; **Django ORM** para trabalhar com dados persistentes dentro de aplicações Django.

**Próximo passo sugerido:** reproduzir o cadastro de produtos numa app Django, criar uma página com formulário e exibir uma tabela com o estoque atualizado.